# Chatbot Pipeline Demo

This notebook demonstrates the **Chatbot Checkpoint Processing Pipeline** that processes chatbot interaction data through a Medallion architecture.

## Architecture: Bronze → Silver → Gold

### 📊 Data Flow

```
Raw CSV Files (Volumes)
 ↓
🥉 Bronze Layer: Auto Loader ingestion
 ↓
🥈 Silver Layer: Data cleansing & enrichment with user/team attribution
 ↓
🥇 Gold Layer: Aggregated business metrics
```

### Catalog & Schema

All datasets are stored in: **`main.chat_history`**

### What This Pipeline Does

* Ingests checkpoint events from CSV files incrementally
* Enriches checkpoints with user and team information
* Produces daily aggregated metrics for user activity and team performance
* Maintains data quality with expectations at each layer

---


In [ ]:
%%sql
-- ## Bronze Layer - Raw Ingestion
SELECT 
 thread_id,
 checkpoint_id,
 type,
 checkpoint,
 metadata,
 ingested_at
FROM main.chat_history.bronze_checkpoints
ORDER BY ingested_at DESC
LIMIT 10;

In [ ]:
%%sql
-- ## Silver - Thread Lookup (User/Team Dimension)
SELECT 
 thread_id,
 user_id,
 team,
 updated_at
FROM main.chat_history.silver_thread_lookup
ORDER BY updated_at DESC
LIMIT 50;

In [ ]:
%%sql
-- ## Silver - Enriched Checkpoints
SELECT 
 thread_id,
 user_id,
 team,
 checkpoint_ts,
 checkpoint_date,
 checkpoint_v,
 metadata_step_int,
 metadata_source
FROM main.chat_history.silver_checkpoints
ORDER BY checkpoint_ts DESC
LIMIT 20;

In [ ]:
%%sql
-- ## Gold - User & Team Metrics
SELECT 
 user_id,
 team,
 checkpoint_date,
 checkpoint_count,
 unique_thread_count,
 first_checkpoint_ts,
 last_checkpoint_ts
FROM main.chat_history.gold_user_team_metrics
WHERE checkpoint_date >= CURRENT_DATE() - INTERVAL 7 DAYS
ORDER BY checkpoint_date DESC, checkpoint_count DESC;

In [ ]:
%%sql
-- ## Gold - Team Summary
SELECT 
 team,
 checkpoint_date,
 checkpoint_count,
 user_count,
 thread_count,
 avg_step
FROM main.chat_history.gold_team_summary
WHERE checkpoint_date >= CURRENT_DATE() - INTERVAL 7 DAYS
ORDER BY checkpoint_date, team;

## Key Insights

This pipeline enables powerful analytics and monitoring capabilities:

### 🎯 User Activity Tracking
* Track individual user interactions across conversations
* Monitor checkpoint frequency and patterns per user
* Identify active vs. inactive users

### 👥 Team-Level Analytics
* Compare team performance and engagement metrics
* Aggregate checkpoint volumes across team members
* Track team growth and activity trends

### 🔍 Checkpoint Monitoring
* System health monitoring through checkpoint patterns
* Detect anomalies in checkpoint sequences
* Version tracking for system upgrades

### 📈 Historical Trend Analysis
* 7-day rolling window for recent activity analysis
* Daily granularity for trend detection
* Step-level metrics for workflow insights

### ✅ Data Quality Enforcement
* **Bronze**: Schema evolution with rescue mode for malformed data
* **Silver**: Expectations drop invalid rows (null timestamps, null IDs)
* **Gold**: Aggregations over validated, clean data only

### 🚀 Performance Optimizations
* Incremental processing via Auto Loader (only new files)
* Change detection on lookup table (skip processing if unchanged)
* Auto-optimized writes for better query performance

---

**Next Steps**: Explore the pipeline code in `/Workspace/Users/chinhang0104@gmail.com/chatbot_pipeline/` or check the README.md for detailed documentation.
